---

# Data Curation

---

## Introduction

The aim of this notebook is to collect and organise medical literature from the PubMed Central (PMC) Open Access Dataset for further NLP analysis on diagnostic criteria detection. Articles are retrieved from PMC Open Access using the NCBI E-utilities API, with searches restricted to titles and abstracts to improve relevance. Full-text articles are then downloaded using the PMC Cloud Service in JATS XML format and converted into structured JSON files to preserve their text, metadata and section information.

The NCBI E-utilities API was selected because it allows biomedical literature to be searched and retrieved automatically using specific search queries. In particular, the ESearch utility returns article identifiers that match the search criteria, allowing relevant PMC articles to be identified without manually searching for each one (Sayers, 2022).

The PMC Cloud Service was used to retrieve full-text articles in JATS XML format. JATS provides a structured representation of scientific articles, preserving information such as sections, subsections, paragraphs and tables. This is useful for the project because diagnostic criteria may be reported in different parts of an article, particularly within the methods, results and tables. The PMC Cloud Service also allows individual full-text articles to be downloaded without requiring an entire dataset, providing an alternative to the rate limitations encountered when using the BioC API.

The downloaded JATS XML articles are then converted into structured JSON files to make them easier to process during further NLP analysis. This allows the original section information and individual passages to be retained while making the article text accessible for further NLP analysis.

## Obtaining the Data

First, we can load in the relevant libraries.

In [ ]:
# import relevant libraries 
import json
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import xml.etree.ElementTree as ET


c:\Users\caral\OneDrive\Documents\SCI501\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We can define the search query. We can define one or more conditions here, separaed by a comma. 

In [14]:
# Define the condition
CONDITION = "aig"

# Define the time period you are interested in.
# NOTE: Recent papers (2026) do not load in, so
# will need to use 2025 and prior. 
SEARCH_PERIOD = '("1900/01/01"[PDAT] : "2025/12/31"[PDAT])'

In [15]:
SEARCH_TERMS = { "AIG": (
        '("autoimmune gastritis"[tiab] OR '
        '"autoimmune atrophic gastritis"[tiab] OR '
        '"autoimmune metaplastic atrophic gastritis"[tiab] OR '
        '"autoimmune chronic atrophic gastritis"[tiab]) '
        'AND "open access"[filter]'
    ) }

In [ ]:
# We can define the search query
# [tiab] limits the search to article titles and abstracts
# The open access filter limits the results to PMC Open Access articles

SEARCH_TERMS = {
    "EPTB_GENERAL": (
        '("extrapulmonary tuberculosis"[tiab] OR '
        '"extra-pulmonary tuberculosis"[tiab] OR '
        '"extra pulmonary tuberculosis"[tiab] OR '
        '"extrapulmonary TB"[tiab]) '
        'AND "open access"[filter]'
    ),

    "TB_MENINGITIS": (
        '("tuberculous meningitis"[tiab] OR '
        '"tuberculosis meningitis"[tiab] OR '
        '"TB meningitis"[tiab]) '
        'AND "open access"[filter]'
    ),

    "PLEURAL_TB": (
        '("pleural tuberculosis"[tiab] OR '
        '"tuberculous pleurisy"[tiab] OR '
        '"tuberculous pleuritis"[tiab] OR '
        '"pleural TB"[tiab]) '
        'AND "open access"[filter]'
    ),

    "LYMPH_NODE_TB": (
        '("tuberculous lymphadenitis"[tiab] OR '
        '"tuberculosis lymphadenitis"[tiab] OR '
        '"lymph node tuberculosis"[tiab] OR '
        '"tuberculous lymphadenopathy"[tiab]) '
        'AND "open access"[filter]'
    )
}


In [ ]:
# We can set up the folders for the downloaded JSON files and
# processed files
JSON_DIR = Path(f"pmc/{CONDITION}/json")
OUTPUT_DIR = Path(f"pmc/{CONDITION}/processed")

# We can create the folders if they do not exist yet
JSON_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NCBI E-utilities API URL used to search PMC and retrieve article IDs
SEARCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/"
    "entrez/eutils/esearch.fcgi"
)

# BioC API URL used to download the full text of each article as JSON
# The {pmcid} will be replaced with the article's PMC ID
BIOC_URL = (
    "https://www.ncbi.nlm.nih.gov/research/bionlp/"
    "RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"
)

# We can define the search query
search_queries = SEARCH_TERMS

# We can create a session to reuse the connection when making API requests
session = requests.Session()

In [17]:
# We can define a function to search the PMC for articles
# matching the search query 
# max_results sets the maximum number of article IDs returned
def search_pmc(query, max_results=100):

    
    query = f"({query}) AND {SEARCH_PERIOD}"
    
    # Send a request to the NCBI E-utilities API
    response = session.get(
        SEARCH_URL,
        params={
            "db": "pmc", # search the PMC database
            "term": query, # use the search terms defined earlier
            "retmax": max_results, # maximum number of article IDs to return
            "retmode": "json", # return the results in JSON format
            "sort": "relevance" # sort articles by relevance
        },
        timeout=30 # stop the request if it takes over 30 seconds
    )

    # check whether the API request was successful
    # raises an error if the server returns a failed HTTP status
    response.raise_for_status()

    # convert the JSON response into a python dictionary 
    # extract the search results from the response
    result = response.json()["esearchresult"]

    # print the total number of articles matching the search query
    print("PMC search matches:", result["count"])

    # return the article IDs with PMC added to each one
    # we will use these IDs to download the full-text articles
    return ["PMC" + x for x in result["idlist"]]

In [18]:
# Set the URL for the PMC Cloud Service
CLOUD_URL = "https://pmc-oa-opendata.s3.amazonaws.com"


# Create a function to extract text from XML elements
# This keeps the text from nested tags such as references and formatting
def extract_text(element):
    return " ".join("".join(element.itertext()).split())


# Create a function to remove XML namespace information
# This makes it easier to identify sections and paragraphs
def tag_name(element):
    return element.tag.split("}")[-1]


# Create a function to download the full-text article from PMC Cloud
# The article will be retrieved in JATS XML format
def download_cloud(pmcid):

    # Search for the available versions of the article
    response = requests.get(
        CLOUD_URL,
        params={
            "list-type": "2",
            "prefix": f"{pmcid}.",
            "delimiter": "/"
        },
        timeout=30
    )
    response.raise_for_status()

    # Extract the available article versions
    root = ET.fromstring(response.content)

    versions = [
        item.text.rstrip("/")
        for item in root.findall(".//{*}CommonPrefixes/{*}Prefix")
        if item.text
    ]

    # Skip articles that are not available through PMC Cloud
    if not versions:
        raise ValueError(f"{pmcid} not available in PMC Cloud")

    # Use the highest available version for this initial test
    version = max(
        versions,
        key=lambda x: int(x.rsplit(".", 1)[1])
    )

    # Download the article metadata
    metadata_url = f"{CLOUD_URL}/metadata/{version}.json"

    response = requests.get(metadata_url, timeout=30)
    response.raise_for_status()
    metadata = response.json()

    # Download the full-text article in JATS XML format
    xml_url = metadata["xml_url"].replace(
        "s3://pmc-oa-opendata/",
        CLOUD_URL + "/"
    )

    response = requests.get(xml_url, timeout=60)
    response.raise_for_status()

    # Return the article metadata and original XML
    return metadata, response.content


# Create a function to convert JATS XML into structured JSON
# This preserves the article sections, paragraphs and tables
def parse_jats(pmcid, xml_content, metadata):

    root = ET.fromstring(xml_content)

    # Create a list to keep track of article passages
    passages = []

    # Keep track of the order of passages in the article
    passage_id = 0

    # Create a function to process each section and subsection
    # This keeps the original section hierarchy
    def process_section(element, section_path):

        nonlocal passage_id

        # Go through the elements within each section
        for child in element:

            tag = tag_name(child)

            # Record the title of the current section
            if tag == "title":
                continue

            # Extract individual paragraphs
            if tag == "p":

                text = extract_text(child)

                if text:
                    passage_id += 1

                    passages.append({
                        "passage_id": passage_id,
                        "section": section_path,
                        "type": "paragraph",
                        "text": text
                    })

            # Extract tables and keep their text
            elif tag == "table-wrap":

                text = extract_text(child)

                if text:
                    passage_id += 1

                    passages.append({
                        "passage_id": passage_id,
                        "section": section_path,
                        "type": "table",
                        "text": text
                    })

            # Process nested subsections
            elif tag == "sec":

                title_element = next(
                    (x for x in child if tag_name(x) == "title"),
                    None
                )

                title = (
                    extract_text(title_element)
                    if title_element is not None
                    else "Untitled section"
                )

                new_path = section_path + [title]

                process_section(child, new_path)

    # Find the article abstract and main body
    for element in root.iter():

        tag = tag_name(element)

        if tag == "abstract":

            # Extract the abstract as a separate passage
            passage_id += 1

            passages.append({
                "passage_id": passage_id,
                "section": ["Abstract"],
                "type": "abstract",
                "text": extract_text(element)
            })

        elif tag == "body":

            # Extract the article body and its sections
            process_section(element, [])

            break

    # Return the article as structured JSON
    return {
        "pmcid": pmcid,
        "metadata": metadata,
        "passages": passages
    }


In [20]:
# Set the number of articles to download per condition
N = 150

# Create a list to keep track of downloaded articles and their details
records = []

# Use a set to keep track of unique PMC IDs and avoid counting duplicates
downloaded = set()

# Set the number of threads to download articles at the same time
MAX_WORKERS = 3


# Create a function to download and save each article
# This allows multiple articles to be processed at the same time
def process_article(pmcid):

    # Set the file paths using the article's PMC ID
    json_path = JSON_DIR / f"{pmcid}.jats.json"
    xml_path = JSON_DIR / f"{pmcid}.xml"

    try:
        # Check whether the article has already been downloaded
        # This avoids downloading the same article again
        if not json_path.exists():

            # Download the full-text article from PMC Cloud
            metadata, xml_content = download_cloud(pmcid)

            # Convert the JATS XML into structured JSON
            # Keep sections and paragraphs for later NLP analysis
            data = parse_jats(pmcid, xml_content, metadata)

            # Check whether any passages were extracted
            if not data["passages"]:
                raise ValueError("No passages extracted")

            # Save the original JATS XML
            # This preserves the full article structure
            xml_path.write_bytes(xml_content)

            # Save the structured article as a JSON file
            # This will be used for BERTopic and LLM analysis
            json_path.write_text(
                json.dumps(data, ensure_ascii=False, indent=2),
                encoding="utf-8"
            )

        # Return the article's PMC ID and saved file location
        return pmcid, str(json_path)

    # Handle errors if an article cannot be downloaded or read
    # Print the reason and continue with the next article
    except (requests.RequestException,
            ValueError,
            ET.ParseError,
            KeyError,
            OSError) as e:
        tqdm.write(f"Skipped {pmcid}: {e}")
        return None


# Go through the search queries
# As it can be multiple items, we will have this
# with a loop
for condition, query in search_queries.items():

    # Show which condition is currently being searched
    print(f"\nSearching for {condition}...")

    # Search PMC to retrieve a list of possible articles
    # Retrieve extra IDs in case some articles are unavailable through Cloud
    candidate_ids = search_pmc(
        query,
        max_results=max(50, N * 3)
    )

    # Remove duplicate IDs while keeping their original order
    candidate_ids = list(dict.fromkeys(candidate_ids))

    # Keep track of how many articles have been successfully processed
    successes = 0

    # Create a pool of threads to download multiple articles at the same time
    # Only submit new articles when a thread becomes available
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        # Keep track of the articles currently being downloaded
        futures = {}
        candidate_iter = iter(candidate_ids)

        # Submit the first articles to the thread pool
        # Only submit as many articles as there are available threads
        for _ in range(min(MAX_WORKERS, N)):
            try:
                pmcid = next(candidate_iter)
                futures[executor.submit(process_article, pmcid)] = pmcid
            except StopIteration:
                break

        # Create a progress bar to show how many articles have been successfully saved
        # This ensures the bar matches the target number of articles
        with tqdm(total=N, desc=condition) as pbar:

            # Process articles as soon as their downloads finish
            while futures and successes < N:

                # Wait for at least one article to finish downloading
                done, _ = wait(futures, return_when=FIRST_COMPLETED)

                # Go through the completed downloads
                for future in done:

                    # Remove the completed article from the active downloads
                    pmcid = futures.pop(future)

                    # Get the download result
                    result = future.result()

                    # Skip articles that could not be downloaded or read
                    if result is not None:

                        # Get the article's PMC ID and saved file location
                        pmcid, file_path = result

                        # Add the PMC ID to the set if it hasn't been counted yet
                        if pmcid not in downloaded:
                            downloaded.add(pmcid)

                        # Record the article's PMC ID, condition and saved file location
                        # This information will be used to create the article manifest
                        records.append({
                            "pmcid": pmcid,
                            "condition": condition,
                            "file": file_path
                        })

                        # Increase the count after successfully processing an article
                        successes += 1

                        # Update the progress bar after successfully processing an article
                        pbar.update(1)

                    # Stop submitting new articles once the target has been reached
                    if successes >= N:
                        break

                    # Submit another article when a thread becomes available
                    # This allows failed downloads to be replaced
                    try:
                        next_pmcid = next(candidate_iter)
                        futures[
                            executor.submit(process_article, next_pmcid)
                        ] = next_pmcid
                    except StopIteration:
                        pass

    # Print how many articles were successfully processed for each condition
    print(f"{condition}: {successes} articles saved")



Searching for AIG...
PMC search matches: 306


AIG:  27%|██▋       | 41/150 [00:33<01:44,  1.05it/s]

Skipped PMC5408474: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


AIG: 100%|██████████| 150/150 [02:32<00:00,  1.02s/it]


AIG: 150 articles saved


In [21]:
# Convert the list of downloaded article records into a DataFrame
# This contains the PMC ID, condition and file location
manifest = pd.DataFrame(records)

# Save the DataFrame as a CSV file
# This allows us to keep track of which articles were downloaded
manifest.to_csv(
    OUTPUT_DIR / "article_manifest.csv",
    index=False  # Don't include the DataFrame index in the CSV
)

# Print a message to confirm the download process has finished
print("\nDownload complete!")

# Show the number of unique articles downloaded
# Articles matching both AIG and PA are only counted once
print("Unique articles:", len(downloaded))

# Show the full folder path where the JSON articles are stored
print("JSON directory:", JSON_DIR.resolve())


Download complete!
Unique articles: 150
JSON directory: C:\Users\caral\OneDrive\Documents\SCI501\SCI501_Project\pmc\aig\json


## Check Downloaded Data

In [22]:
# Get a list of all JSON files saved in the download folder
json_files = list(JSON_DIR.glob("*.jats.json"))

# Print the total number of JSON articles
print("Downloaded JSON files:", len(json_files))

# Show an example file name to check that articles were saved (and saved correctly)
# If no files found, print "None"
print("Example filename:", json_files[0].name if json_files else "None")

# Go through each condition and count the articles recorded
# This shows how many articles were collected for each condition
for condition in search_queries:

    condition_files = [
        record for record in records
        if record["condition"] == condition
    ]

    print(f"{condition}: {len(condition_files)} articles")

Downloaded JSON files: 152
Example filename: PMC10052490.jats.json
AIG: 150 articles


In [23]:
# Open the first downloaded JATS JSON article
# Use UTF-8 encoding to read the text correctly as the articles
# may have special characters
with open(json_files[0], encoding="utf-8") as f:

    # Load the JSON file into a Python object
    # So we can inspect the article's text, metadata and structure
    article = json.load(f)

# Display the article to see how the JATS data is organised
article

{'pmcid': 'PMC10052490',
 'metadata': {'pmcid': 'PMC10052490',
  'version': 1,
  'pmid': 36967703,
  'doi': '10.1177/03000605231162451',
  'mid': None,
  'title': 'Autoimmune gastritis with a gastric hamartomatous inverted polyp and two hyperplastic polyps: a case report',
  'citation': 'J Int Med Res. 2023 Mar 27;51(3):03000605231162451. doi: 10.1177/03000605231162451',
  'is_pmc_openaccess': True,
  'is_manuscript': False,
  'is_historical_ocr': False,
  'is_retracted': False,
  'license_code': 'CC BY-NC',
  'text_url': 's3://pmc-oa-opendata/PMC10052490.1/PMC10052490.1.txt?md5=4f96b5c0d015d41d7fb08d50f1faf60c',
  'xml_url': 's3://pmc-oa-opendata/PMC10052490.1/PMC10052490.1.xml?md5=9c5e66a062fe6931cb7d88e999b3acfd',
  'pdf_url': 's3://pmc-oa-opendata/PMC10052490.1/PMC10052490.1.pdf?md5=1c3f07a6bf40281ff8a69bdf3cee0b2d',
  'media_urls': ['s3://pmc-oa-opendata/PMC10052490.1/10.1177_03000605231162451-fig1.jpg?md5=6bc50ae499cdc457a9c19ae906354521',
   's3://pmc-oa-opendata/PMC10052490.1/1

## Sources

---

Sayers, E. W. (2022). A general introduction to the E-utilities. In Entrez programming utilities help. National Center for Biotechnology Information. https://www.ncbi.nlm.nih.gov/books/NBK25497/

National Center for Biotechnology Information. (n.d.). PubMed Central (PMC) article datasets: Full-text biomedical and life sciences journal articles on AWS [Data set]. AWS Open Data Registry. https://registry.opendata.aws/ncbi-pmc